## Idea

1. Since stock ticker is at most 6 characters, we can achieve perfect hashing by mapping each entire ticker to an integer within [1 ... 2^6] = [1 ... 64]. 


2. We want 0 to mean “empty spot” or padding in case the stock has < 6 chars. Example: "AAPL" is only 4 chars, but max length is 6, so we store it like: `A A P L pad pad`


3. Since the max range is 64, this is equivalent to: ticker (str) -> base-64 integer:

    `code = c1 * 64^5 + c2 * 64^4 + c3 * 64^3 + c4 * 64^2 + c5 * 64 + c6`

    -> This is equivalent to: from left to right, bit-shift `<<` fron highest to lowest power

In [2]:
class TickerEncoder:
    def __init__(self):
        chars = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789."
        self.char_to_val = {ch: i + 1 for i, ch in enumerate(chars)}
        self.pad = 0
        self.max_len = 6

    def encode(self, ticker: str) -> int:
        assert len(ticker) <= self.max_len
        ticker = ticker.upper()

        code = 0

        for i in range(self.max_len):
            """ code = code * 64 + next_char_value """
            code <<= 6  # code = code * 2^6 = 64
            code |= (
                self.char_to_val[ticker[i]] 
                if (i < len(ticker)) else self.pad
            )           # code = code + next_char_value

        return code

    def decode(self, code: int) -> str:
        answers = []

        """ Get right char one at a time """
        vals = []
        for _ in range(self.max_len):
            vals.append(code & 0b111111) # code % 2^6 (since six 1's) 
            code >>= 6                   # code //= 2^6 = 64
    
        vals.reverse()
        
        val_to_char = {v: k for k, v in self.char_to_val.items()}
        for v in vals:
            if v == self.pad: continue
            else: answers.append(val_to_char[v])

        return "".join(answers)


# example
enc = TickerEncoder()
stocks = ["AAPL", "MSFT", "BRK.B"]

for stock in stocks:
    assert enc.decode(enc.encode(stock)) == stock